# model_inference.ipynb

Loads the **best registered model** (XGBoost in my runs) directly from the
MLflow Model Registry and runs `predict_proba` on the **raw** test set.
No manual preprocessing — the saved sklearn `Pipeline` does it all.

## 0. Setup

In [ ]:
import os
import sys
import pandas as pd
import numpy as np

sys.path.append("..")
from src.preprocessing import load_raw

import mlflow
import mlflow.sklearn

try:
    import dagshub
    dagshub.init(
        repo_owner=os.environ.get("DAGSHUB_USER", "your-username"),
        repo_name=os.environ.get("DAGSHUB_REPO", "ieee-fraud-detection"),
        mlflow=True,
    )
except Exception as e:
    print("dagshub init skipped:", e)
    mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", "file:./mlruns"))


## 1. Load raw test set

Notice we do NOT preprocess anything here — the pipeline pulled from the
registry already contains the cleaning + FE + FS steps.

In [ ]:
DATA_PATH = os.environ.get("IEEE_DATA", "../data")
_, test = load_raw(DATA_PATH)
print("test:", test.shape)

X_test = test.drop(columns=["TransactionID"])


## 2. Pull best model from the Registry

I registered the XGBoost pipeline as `ieee_fraud_best` after comparing CV scores.
We use the `Production` stage so the inference notebook always uses the latest promoted model.

In [ ]:
MODEL_NAME = os.environ.get("MODEL_NAME", "ieee_fraud_best")
STAGE = os.environ.get("MODEL_STAGE", "Production")

model_uri = f"models:/{MODEL_NAME}/{STAGE}"
print("loading", model_uri)
pipe = mlflow.sklearn.load_model(model_uri)
print(pipe)


## 3. Predict on raw test data

In [ ]:
proba = pipe.predict_proba(X_test)[:, 1]
print("predictions:", proba.shape, "  range:", proba.min(), proba.max())


## 4. Build the Kaggle submission

In [ ]:
sub = pd.DataFrame({
    "TransactionID": test["TransactionID"],
    "isFraud": proba,
})
sub.to_csv("submission.csv", index=False)
sub.head()


In [ ]:
# kaggle command (run in terminal, not here):
# kaggle competitions submit -c ieee-fraud-detection -f submission.csv -m "best XGBoost pipeline"


## 5. (Optional) — register a new model from a run

Useful when comparing models. After the experiment notebooks finish, find the
best run by CV AUC and register its pipeline.

In [ ]:
# example — only run if you want to register a fresh model
# from mlflow.tracking import MlflowClient
#
# client = MlflowClient()
# best_run_id = "PASTE_RUN_ID_FROM_BEST_FINALFIT"
# result = mlflow.register_model(
#     model_uri=f"runs:/{best_run_id}/pipeline",
#     name=MODEL_NAME,
# )
# client.transition_model_version_stage(
#     name=MODEL_NAME, version=result.version, stage="Production",
#     archive_existing_versions=True,
# )
